In [1]:
# Assignment 01
from pyspark.sql import SparkSession
import json

# 1. Initialize SparkSession
spark = SparkSession.builder \
    .appName("Generate Parquet Data") \
    .config("spark.sql.parquet.enableStats", "true") \
    .getOrCreate()

dict = {"num_rows": "", "format": "", "status": ""}

df = spark.read.parquet("/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/FirstProject/src/leetcode/large_practice_data.parquet")

df.show(5)

if df is not None:
    dict["num_rows"] = df.count()
    
    # Get the source path from the Spark plan
    logical_plan = df._jdf.queryExecution().logical()
        
    source_info = logical_plan.toString()

    if "Parquet" in source_info:
        file_format = "parquet"
    elif "CSV" in source_info:
        file_format = "csv"
    else:
        file_format = "unknown"
    print(f"Detected Format: {file_format}")
    dict["format"] = file_format

    dict["status"] = "success"



print (dict)

#create a json file to store the metadata

with open("/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/Feb_2026/metadata.json", "w") as f:
    json.dump(dict, f, indent=4)


 
spark.stop()

26/02/17 12:50:16 WARN Utils: Your hostname, Yashwanths-Mac-mini.local resolves to a loopback address: 127.0.0.1; using 192.168.1.3 instead (on interface en1)
26/02/17 12:50:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/17 12:50:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+---+--------------+-------+-----------+--------------------+-----+----------------+------------------+------------------+-----------+-----------------+---------------+---------------+---------------+
| id|transaction_id|user_id|   category|           timestamp| name|           email|            amount|             score|category_id|         discount|date_mm_dd_yyyy|date_dd_mm_yyyy|date_yyyy_mm_dd|
+---+--------------+-------+-----------+--------------------+-----+----------------+------------------+------------------+-----------+-----------------+---------------+---------------+---------------+
|900|           900|   4853|       null|2023-02-13 15:35:...|Alice|            null| 376.2585170603643|              null|       null|             null|     02-13-2023|           null|           null|
|901|           901|   5779|Electronics|2023-08-25 04:44:...|Alice|            null|105.43840119496156| 42.47771204426548|          8|             null|     08-25-2023|     25-08-2023|           n

In [12]:
# Assignment 02
''' This should read the record.json file along with input file .
    If the status in record.json is valid then perform the below activity else exit the program with exception as "Data is not correct" 
    The Names should be capitalised .
    The date should be uniform ( DD-MM-YYYY)
    Null should be replaced with Unknown for text field , 0 for numeric fields.'''

import json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType, IntegerType, FloatType, DoubleType,
    DateType, TimestampType, BooleanType
)


spark = SparkSession.builder.appName("Read Parquet Data").getOrCreate()


with open("/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/Feb_2026/metadata.json", "r") as f:
    metadata = json.load(f)

if metadata["status"] == "success":
    
    df = spark.read.parquet(
        "/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/FirstProject/src/leetcode/large_practice_data.parquet"
    )
    df.show(10, truncate=False)
    
    for field in df.schema.fields:
        col_name = field.name
        dtype = field.dataType

       
        if isinstance(dtype, StringType):
            df = df.withColumn(col_name, F.initcap(F.col(col_name)))

           
            if "date" in col_name.lower():
                df = df.withColumn(
                    col_name,
                    F.date_format(
                        F.coalesce(
                            F.to_date(F.col(col_name), "MM-dd-yyyy"),
                            F.to_date(F.col(col_name), "dd-MM-yyyy"),
                            F.to_date(F.col(col_name), "yyyy-MM-dd")
                        ),
                        "dd-MM-yyyy"
                    )
                )

       
        elif isinstance(dtype, (DateType, TimestampType)):
            df = df.withColumn(col_name, F.date_format(F.col(col_name), "dd-MM-yyyy"))

    # Replace nulls for text and numeric fields

    for field in df.schema.fields:
        col_name = field.name
        dtype = field.dataType

        if isinstance(dtype, StringType):
            # Replace null strings with "Unknown"
            df = df.withColumn(col_name, F.coalesce(F.col(col_name), F.lit("Unknown")))

        elif isinstance(dtype, (IntegerType, FloatType, DoubleType)):
            # Replace null numbers with 0
            df = df.withColumn(col_name, F.coalesce(F.col(col_name), F.lit(0)))

        elif isinstance(dtype, BooleanType):
            # Replace null booleans with False
            df = df.withColumn(col_name, F.coalesce(F.col(col_name), F.lit(False)))

  
    df.show(10, truncate=False)
else:
    raise Exception("Data is not correct")


spark.stop()

+---+--------------+-------+-----------+--------------------------+-----+----------------+------------------+------------------+-----------+------------------+---------------+---------------+---------------+
|id |transaction_id|user_id|category   |timestamp                 |name |email           |amount            |score             |category_id|discount          |date_mm_dd_yyyy|date_dd_mm_yyyy|date_yyyy_mm_dd|
+---+--------------+-------+-----------+--------------------------+-----+----------------+------------------+------------------+-----------+------------------+---------------+---------------+---------------+
|900|900           |4853   |null       |2023-02-13 15:35:59.919691|Alice|null            |376.2585170603643 |null              |null       |null              |02-13-2023     |null           |null           |
|901|901           |5779   |Electronics|2023-08-25 04:44:40.928345|Alice|null            |105.43840119496156|42.47771204426548 |8          |null              |08-25-202